# Purple AI V1.0 — trening od zera w Google Colab

Notebook pobiera około 11 000 polskich dokumentów, trenuje nasz własny model z losowych wag i po treningu wysyła checkpoint, konfigurację oraz raport bezpośrednio do repozytorium modelu Hugging Face `Kamiljdds/purpleaiV1.0`.

Do wysłania potrzebny jest token Hugging Face z uprawnieniem **Write**. Wkleisz go tylko w bezpiecznym polu Colab (albo zapiszesz jako sekret `HF_TOKEN`) — nie wysyłaj tokenu na czacie ani do GitHuba.

Dane: `SlayerLab/polish-dynaword`, wersja `v0.2.5`. Notebook zachowuje manifest i informację o licencji; sprawdź kartę zbioru i warunki źródeł przed dalszą dystrybucją.

In [ ]:
import os, sys, json, shutil, zipfile, subprocess
from pathlib import Path

REPO = Path('/content/PurpleAI')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/kaielk09/Ai.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

%pip -q install datasets==3.3.2 huggingface_hub

print('Repo:', REPO)
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'brak — Colab może trenować wolno na CPU')

## 1. Pobierz około 11 000 polskich tekstów

To jest **pretraining językowy**: model uczy się przewidywać kolejne bajty tekstu. Sam zbiór dokumentów nie uczy jeszcze poprawnego formatu asystenta ani wykonywania poleceń.

In [ ]:
from datasets import load_dataset
from collections import Counter

TARGET_ROWS = 11_000
MAX_CHARS_PER_DOC = 5_000
ALLOWED_SOURCES = {
    'wikibooks', 'wikivoyage', 'wolne_lektury',
    'wiktionary_examples', 'global_voices', 'govpl',
    'samorzad_gov_pl', 'nkjp1m', 'wikiquote', 'wikinews'
}

stream = load_dataset(
    'SlayerLab/polish-dynaword',
    revision='v0.2.5',
    split='train',
    streaming=True,
).shuffle(seed=321, buffer_size=20_000)

texts = []
source_counts = Counter()
scanned = 0
for row in stream:
    scanned += 1
    source = str(row.get('source', '')).strip()
    text = row.get('text', '')
    if source not in ALLOWED_SOURCES or not isinstance(text, str):
        if scanned >= 500_000:
            break
        continue
    text = ' '.join(text.split()).strip()
    if len(text) < 100:
        continue
    texts.append(text[:MAX_CHARS_PER_DOC])
    source_counts[source] += 1
    if len(texts) >= TARGET_ROWS:
        break
    if scanned >= 500_000:
        break

if len(texts) < TARGET_ROWS:
    raise RuntimeError(f'Pobrano tylko {len(texts)} dokumentów po sprawdzeniu {scanned} rekordów. Nie uruchamiam treningu na niepełnym zbiorze.')

data_dir = REPO / 'training'
data_dir.mkdir(parents=True, exist_ok=True)
data_path = data_dir / 'data.txt'
data_path.write_text('\n\n'.join(texts), encoding='utf-8')

manifest = {
    'dataset': 'SlayerLab/polish-dynaword',
    'revision': 'v0.2.5',
    'dataset_license': 'CC-BY-SA-4.0',
    'rows': len(texts),
    'sources': dict(source_counts),
    'max_chars_per_document': MAX_CHARS_PER_DOC,
    'note': 'Training subset; retain attribution and comply with upstream per-source rights.'
}
(REPO / 'training_data_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Gotowe:', len(texts), 'dokumentów')
print('Źródła:', dict(source_counts))
print('Rozmiar data.txt:', round(data_path.stat().st_size / 1_000_000, 2), 'MB')

## 2. Ustawienia i trening

Checkpoint będzie zapisywany co 100 kroków. Jeśli Colab przerwie sesję, ponowne uruchomienie komórek może wznowić trening z ostatniego zapisu, o ile katalog `/content/purple_ai_output` nadal istnieje. Zapisuj kopię checkpointu, gdy trening trwa.

In [ ]:
OUTPUT = Path('/content/purple_ai_output')
OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ['PURPLE_DATA'] = str(data_path)
os.environ['PURPLE_CHECKPOINT_DIR'] = str(OUTPUT)
os.environ['PURPLE_STEPS'] = '5000'
os.environ['PURPLE_BATCH_SIZE'] = '8'
os.environ['PURPLE_BLOCK_SIZE'] = '512'
os.environ['PURPLE_EVAL_EVERY'] = '100'
os.environ['PURPLE_LR'] = '0.0003'

train_script = REPO / 'training' / 'train_from_scratch.py'
subprocess.run([sys.executable, str(train_script)], cwd=str(REPO), check=True)

checkpoint = OUTPUT / 'purple_ai_latest.pt'
if not checkpoint.is_file():
    raise FileNotFoundError('Nie znaleziono checkpointu — trening nie zakończył się poprawnie.')
print('Checkpoint gotowy:', checkpoint, 'Rozmiar:', round(checkpoint.stat().st_size / 1_000_000, 2), 'MB')

## 3. Sprawdź checkpoint i wyślij wynik na Hugging Face

Po udanym treningu notebook sprawdza checkpoint, tworzy raport i atrybucję, a następnie wysyła pliki do `Kamiljdds/purpleaiV1.0` w folderze `training-runs/colab-11k`. Dane źródłowe `data.txt` nie są wysyłane.

Jeśli repozytorium jest prywatne, token musi mieć do niego uprawnienia zapisu. Nie umieszczaj tokenu w kodzie ani w publicznym notebooku.

## Po treningu

Najpierw wynik jest wysyłany do repozytorium Hugging Face `Kamiljdds/purpleaiV1.0`, do folderu `training-runs/colab-11k`. Dopiero potem notebook pobiera lokalną kopię ZIP. Nie wysyłamy surowego `data.txt` ani nie zapisujemy wag w repozytorium GitHub.

To pierwszy etap pretrainingu, a nie gotowy asystent. Sam niski loss nie potwierdza jeszcze jakości odpowiedzi.

In [ ]:
import torch, json, getpass
from huggingface_hub import HfApi

OUTPUT = Path('/content/purple_ai_output')
checkpoint = OUTPUT / 'purple_ai_latest.pt'
if not checkpoint.is_file():
    raise FileNotFoundError('Nie znaleziono checkpointu — trening nie zakończył się poprawnie.')

saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
report = {
    'model': 'Purple AI V1.0',
    'training_completed_steps': int(saved.get('step', 0)),
    'config': saved.get('config', {}),
    'last_training_loss': saved.get('loss'),
    'last_validation_loss': saved.get('validation_loss'),
    'dataset_rows': TARGET_ROWS,
    'dataset': 'SlayerLab/polish-dynaword@v0.2.5',
    'training_type': 'from_random_initialization'
}
(OUTPUT / 'training_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
(OUTPUT / 'training_data_manifest.json').write_text((REPO / 'training_data_manifest.json').read_text(encoding='utf-8'), encoding='utf-8')
(OUTPUT / 'DATASET_ATTRIBUTION.txt').write_text(
    'Dataset: SlayerLab/polish-dynaword, revision v0.2.5.\\n'
    'Dataset card: https://huggingface.co/datasets/SlayerLab/polish-dynaword\\n'
    'Corpus license listed by dataset: CC-BY-SA-4.0; underlying sources may have individual terms.\\n'
    'Retain attribution and review upstream terms before redistribution.\\n', encoding='utf-8')

print(json.dumps(report, ensure_ascii=False, indent=2))

# Pobierz token z sekretu Colab HF_TOKEN albo użyj ukrytego pola wejścia.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Wklej token Hugging Face z uprawnieniem Write (nie będzie wyświetlony): ')
if not HF_TOKEN:
    raise RuntimeError('Brak tokenu HF. Zapisz sekret HF_TOKEN w Colab albo uruchom komórkę ponownie.')

REPO_ID = 'Kamiljdds/purpleaiV1.0'
api = HfApi(token=HF_TOKEN)
# Sprawdź, czy konto ma dostęp do istniejącego repozytorium.
api.repo_info(repo_id=REPO_ID, repo_type='model')
api.upload_folder(
    repo_id=REPO_ID,
    repo_type='model',
    folder_path=str(OUTPUT),
    path_in_repo='training-runs/colab-11k',
    commit_message='Upload Purple AI V1.0 training checkpoint and report',
    ignore_patterns=['*.tmp', '*.log']
)
print('Wysłano wynik treningu do:', f'https://huggingface.co/{REPO_ID}/tree/main/training-runs/colab-11k')

# Dodatkowa kopia ZIP do pobrania na urządzenie.
bundle = Path('/content/Purple_AI_V1_training_result.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT.iterdir():
        if p.is_file(): z.write(p, p.name)
from google.colab import files
files.download(str(bundle))